<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/04-agents/01-agent-loop-from-scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agent Loop from Scratch

**Goal:** Build a working agent loop with raw API calls — no framework.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks) — the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).


## Setup

Each notebook is self-contained, so the next two cells stand it up from scratch:

1. **Install dependencies.** The `aien` package (this repo) carries the shared setup helper and pulls in the `groq` client — the only dependency this notebook needs.
2. **Load your API key.** Get a free key at [console.groq.com](https://console.groq.com/) (no credit card). In Colab, add it via the **key icon** in the left sidebar → **Add new secret**, name it exactly `GROQ_API_KEY`, paste the value, and toggle **Notebook access** on. Running locally instead? Set `GROQ_API_KEY` as an environment variable.

(Full walkthrough and model-picking guidance live in [00-setup/00-environment.ipynb](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/00-setup/00-environment.ipynb).)

In [ ]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git"

In [ ]:
from aien import setup

# Loads GROQ_API_KEY (Colab Secrets or local env var) and returns a ready
# Groq client. Pass model=... to override the default; if a call later 404s,
# list available models — see 00-setup/00-environment.ipynb.
client, MODEL = setup()

## Why a loop — and an agent is just a while loop

Here's a task for this notebook: *"Read all the expense notes in the workspace, add them up, and write a summary file with the total."* A single model call can't do it. The model doesn't know which files exist, can't read them, can't run the arithmetic, and can't write the result — and even with tools (notebook 01-02), *one* call only gets you *one* round of tool use. This task needs a chain: list files → read each → calculate → write → report, where every step depends on what the last one returned.

An **agent** is what runs that chain. Strip away the marketing and it's four things:

1. **A model** that can request tool calls.
2. **Tools** — plain functions you expose with a name, a description, and a schema.
3. **A loop**: call the model; if it asked for tools, run them, append the results, go again.
4. **Stop conditions**: the model stops asking for tools, or you hit a turn limit.

That's it — the model drives, the loop keeps handing control back to it until the task is done. The whole thing fits in about 40 lines of Python, and building it once by hand is the fastest way to stop being mystified by agent frameworks. We'll do exactly that here, then run it on the multi-step task above.

## The tools

We need tools that are safe to run in Colab with no external services. Two will do:

- A tiny **in-memory filesystem** — just a dict — with `list_files`, `read_file`, and `write_file`.
- A **calculator** that evaluates arithmetic expressions.

Each tool has two halves: the *implementation* (a Python function) and the *definition* (name + description + JSON schema) that the model sees. The model never sees your code — only the definition. Keep that in mind; it becomes the whole subject of notebook 02.

In [ ]:
import json
import re

# A fake filesystem: path -> content. Safe, inspectable, resettable.
FS = {
    "notes/monday.md": "Team lunch: $42.50\nTaxi to client site: $18.00\n",
    "notes/tuesday.md": "Conference tickets, 2 x $150.00 each\n",
    "notes/wednesday.md": "Cloud credits top-up: $75.25\nCoffee for the workshop: $23.10\n",
}

def list_files():
    return "\n".join(sorted(FS)) if FS else "(no files)"

def read_file(path):
    if path not in FS:
        return f"Error: no file at '{path}'. Call list_files to see what exists."
    return FS[path]

def write_file(path, content):
    FS[path] = content
    return f"Wrote {len(content)} characters to {path}."

def calculator(expression):
    # Only digits, operators, parens, dots, spaces -- no names, no attribute access.
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expression):
        return "Error: only numbers and + - * / ( ) are allowed."
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

IMPLS = {
    "list_files": list_files,
    "read_file": read_file,
    "write_file": write_file,
    "calculator": calculator,
}

def execute_tool(name, args):
    fn = IMPLS.get(name)
    if fn is None:
        return f"Error: unknown tool '{name}'."
    try:
        return str(fn(**args))
    except TypeError as e:
        return f"Error: bad arguments for {name}: {e}"

# Tool definitions in the OpenAI-compatible format Groq uses:
# each is {"type": "function", "function": {name, description, parameters}}.
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List every file path in the workspace, one per line. "
                           "Use this first if you don't know which files exist.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read the full contents of one file. Returns the text, "
                           "or an error if the path doesn't exist.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string", "description": "Exact path from list_files."}},
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Create or overwrite a file with the given content. "
                           "Returns a confirmation.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string"},
                    "content": {"type": "string"},
                },
                "required": ["path", "content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate an arithmetic expression using + - * / and parentheses, "
                           "e.g. '42.50 + 18.00'. Use this for all math instead of computing "
                           "in your head. Returns the numeric result.",
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string"}},
                "required": ["expression"],
            },
        },
    },
]


## The loop

The API is stateless: every request carries the full conversation so far, and the `messages` list *is* the agent's entire state. Each iteration:

1. Send `messages` to the model.
2. If `finish_reason == "tool_calls"`, the model's reply contains one or more tool calls. Append the reply as-is, execute every requested tool, and send each result back as its own `role: "tool"` message (matched by `tool_call_id`).
3. Otherwise the model is done — return its text.

The `max_turns` bound is not optional decoration. Every turn is a paid API call, so an unbounded loop is an unbounded bill. Every loop in this repo carries one.

We also print a trace of every turn. The trace is the pedagogy — it's how you see the model plan.

In [ ]:
def run_agent(task, tools, max_turns=10):
    """An agent: a model, some tools, a while loop, and stop conditions."""
    messages = [{"role": "user", "content": task}]

    for turn in range(1, max_turns + 1):
        response = client.chat.completions.create(
            model=MODEL,
            max_tokens=1024,
            tools=tools,
            messages=messages,
        )
        choice = response.choices[0]
        msg = choice.message

        # Trace: any text the model produced this turn (its "thinking out loud")
        if msg.content and msg.content.strip():
            print(f"[turn {turn}] model: {msg.content.strip()[:200]}")

        # Stop condition #1: the model didn't ask for a tool -- it's done.
        if choice.finish_reason != "tool_calls":
            return msg.content or ""

        # Append the assistant turn unchanged (it carries the tool_calls list).
        messages.append(msg)

        # Execute every requested tool; append each result as its own tool message.
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            result = execute_tool(tc.function.name, args)
            print(f"[turn {turn}] tool : {tc.function.name}({tc.function.arguments[:120]})")
            print(f"[turn {turn}]   -> {result.strip()[:120]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result,
            })

    # Stop condition #2: turn limit. Fail loud, not silent.
    return f"Stopped: hit max_turns={max_turns} before the task finished."


That's the entire agent. Two stop conditions, one trace, no framework.

## Run it

The task below can't be done in one shot: the model has to discover the files, read them, do math, and write a result — a chain of 3–5 tool calls whose later steps depend on earlier results.

In [ ]:
result = run_agent(
    "Read all the notes files in the workspace, add up every expense using the "
    "calculator, and write a file called summary.md that states the total. "
    "Then tell me the total.",
    tools=TOOLS,
)
print("\n=== final answer ===")
print(result)


In [ ]:
# The side effect is real: the agent wrote into our fake filesystem.
print(FS.get("summary.md", "(summary.md was not written)"))


## What the trace shows

Run the cells above and read the trace top to bottom. A few things worth noticing:

- **The model plans across turns without being told to.** It typically lists files first, reads them, computes with the calculator, then writes. Nobody scripted that order; it falls out of the loop plus decent tool descriptions.
- **Tuesday's note requires actual reasoning**: "2 x $150.00 each" isn't a number you can copy — watch how the expression it sends to the calculator handles it.
- **The `messages` list is the agent's entire state.** There is no hidden memory. If you printed `messages` at the end you'd see the complete run: task, tool calls, results, answer.
- **Every turn re-sends the full history.** Turn 5's request contains turns 1–4 verbatim. Context growth is cost growth — roughly quadratic over a long run — which is why notebook 03 puts budgets on this loop, and why tool *output size* matters so much in notebook 02.

Run it twice and the traces will differ in the details (order of reads, exact expression) while converging on the same answer. That non-determinism is normal; it's also why production agents need the guardrails we add later.

## What frameworks add

LangGraph, CrewAI, the OpenAI Agents SDK, Claude's tool runner — at their core, every one of them runs this same loop. What they add on top:

- **Persistence** — checkpointing `messages` so a run survives a process restart.
- **Observability** — structured traces instead of our `print` statements.
- **Handoffs / multi-agent wiring** — routing between several loops with different tools.
- **Retries, rate limiting, human-approval hooks** — operational plumbing.

All useful; none magic. Having built the loop by hand, you can now evaluate a framework on exactly those terms: what does it persist, what does it show me, what does it wire together — and what does it hide?

For the production and multi-agent view of this loop — queues, checkpointing, orchestration — see the systems-level companion: [Agent Orchestration walkthrough](https://www.calm.rocks/resources/prepare-interview/system-design/agent-orchestration-walkthrough/).

## Exercises

1. **Add a `delete_file` tool** (implementation, definition, and dispatch entry), then give the agent a task that requires it — e.g. "consolidate all notes into one file and delete the originals." Watch the trace to confirm the ordering it picks.
2. **Break a description on purpose.** Change `calculator`'s description to just "Does math." and remove the example. Re-run the same task three times — does the model still use it, or start doing arithmetic in its head? (This is the notebook 02 thesis in miniature.)
3. **Print the token cost curve.** Capture `response.usage.prompt_tokens` each turn and print it. Confirm that input tokens grow every turn even though your task text never changed — then estimate the cost of a 50-turn run at current per-token pricing.
4. **Return the trace, not just the answer.** Change `run_agent` to also return a list of `(turn, tool_name, args, result)` tuples, and write five lines of code that answer: how many turns, how many tool calls, which tool was called most?